# DataMate — Qwen7B · 890 mẫu gọn · Kaggle T4 ×2
Input mới: `datamate_qwen7b_890_portable_train_eval_kaggle_input.zip` + weights Qwen2.5-Coder-7B-Instruct.

Dataset chỉ lưu `id`, `business_id`, `code`. Notebook tra câu hỏi, đọc README/schema từ `training/train` trong Input read-only và dựng prompt bằng source hiện tại. Đường dẫn máy không được đưa vào prompt; không đọc gold khi dựng dữ liệu SFT.

Train FP16 LoRA/FSDP, upload adapter và xác minh file trên HF **trước** khi cài runtime/chạy pilot. Upload cần Internet và Kaggle Secret `HF_TOKEN` có quyền write. Checkpoint nền được giới hạn snapshot để tránh tăng hàng đợi chiếm đĩa.

**Adapter cũ đã train xong:** dùng notebook `upload_trained_qwen7b_to_hf.ipynb` để upload riêng, không chạy lại training. Khi muốn chấm trong notebook này, đặt `EVAL_ADAPTER_DIR`, chạy setup/source/model và bỏ qua các cell smoke/train.

Pilot 30 câu có 14 ID giao với SFT; đây không phải điểm đánh giá hoàn toàn trên câu chưa học.

## 1. Cấu hình phiên chạy


In [ ]:
from pathlib import Path
import os, json, hashlib, shutil, subprocess, sys, math

MODEL_DIR = ''  # Để trống để tự tìm; nếu nhiều model thì nhập đường dẫn Instruct trong /kaggle/input.
RUN_NAME = 'qwen7b_890_portable_fp16_lora'
MAX_SEQ_LEN = 8192
EPOCHS = 3.0
LEARNING_RATE = 1e-4
GRAD_ACCUM = 4  # batch 1/GPU × 2 GPU × 4 = 8 mẫu/optimizer step.
RESUME_FROM_CHECKPOINT = ''  # 'latest' hoặc đường dẫn checkpoint-N từ Output phiên cũ.
EVAL_ADAPTER_DIR = ''  # Nếu đã train: điền đường dẫn final_adapter từ Kaggle Input/Output, rồi bỏ qua smoke/train.
HF_REPO_NAME = 'Zalo-challenge'  # Repo private dưới tài khoản HF_TOKEN; để '' nếu chỉ cần log local.

EXPECTED_DATA_SHA256 = '038c55a8b53c0b8e4f614e51f341fbc839bd0b7162dc6a79985d0c94be2ab1fb'
WORK = Path('/kaggle/working/datamate_qwen7b_890')
RUN_ROOT = WORK / 'runs' / RUN_NAME
WORK.mkdir(parents=True, exist_ok=True)
os.environ.update(USE_TF='0', USE_TORCH='1', TOKENIZERS_PARALLELISM='false',
                  NCCL_P2P_DISABLE='1', NCCL_IB_DISABLE='1', OMP_NUM_THREADS='2', PYTHONUNBUFFERED='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=True)
else:
    print('Không có GPU: chỉ chạy upload; bước train cần T4 x2.')

## 2. Kiểm tra bundle mới và dataset ba trường

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

candidates = []
for candidate in Path('/kaggle/input').rglob('bundle_manifest.json'):
    manifest = json.loads(candidate.read_text(encoding='utf-8'))
    if (manifest.get('bundle') == 'datamate-qwen7b-sft890-2t4'
            and manifest.get('revision') == 'portable-sft-hf-v3'):
        candidates.append((candidate.parent, manifest))
if len(candidates) != 1:
    raise RuntimeError(f'Cần đúng 1 Input bundle portable-sft-hf-v3, tìm được {len(candidates)}. Upload ZIP portable mới; bỏ Input cũ khỏi phiên này.')
bundle, manifest = candidates[0]
for relative, expected in manifest['files'].items():
    assert sha256(bundle / relative) == expected, f'Bundle checksum mismatch: {relative}'
assert manifest['dataset_sha256'] == EXPECTED_DATA_SHA256
PROJECT = WORK / 'source'
PROJECT.mkdir(parents=True, exist_ok=True)
# Raw training/pilot data stay in Input; copy only source and the compact dataset.
for name in ['src', 'scripts', 'configs', 'dataset']:
    shutil.copytree(bundle / name, PROJECT / name, dirs_exist_ok=True)
shutil.copy2(bundle / 'requirements-kaggle-sft.txt', PROJECT / 'requirements-kaggle-sft.txt')
COMPACT_DATASET = PROJECT / 'dataset/verified_sft_dataset.jsonl'
assert sha256(COMPACT_DATASET) == EXPECTED_DATA_SHA256
records = [json.loads(line) for line in COMPACT_DATASET.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(records) == len({row['id'] for row in records}) == 890
assert all(set(row) == {'id', 'business_id', 'code'} for row in records)
TRAIN_DATA = bundle / 'training/train'
assert (TRAIN_DATA / 'questions.jsonl').is_file()
DATASET = WORK / 'prepared/sft_train_ready.jsonl'
os.chdir(PROJECT)
print('Source:', PROJECT)
print('Compact dataset:', COMPACT_DATASET, '| records:', len(records))
print('Train context (read-only):', TRAIN_DATA)
print('Free working disk:', round(shutil.disk_usage(WORK).free / 2**30, 2), 'GiB')

## 3. Cài đúng phiên bản thư viện, giữ CUDA PyTorch của Kaggle


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-r', 'requirements-kaggle-sft.txt'], check=True)
# Optional torchao có thể được cài sẵn với version không tương thích PEFT; LoRA FP16 không dùng nó.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', """
import torch, transformers, peft, accelerate
from packaging.version import Version
from torch import nn
from peft import LoraConfig, get_peft_model
assert Version(torch.__version__.split('+')[0]) >= Version('2.6')
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Chọn T4 x2 trong Settings'
assert all(torch.cuda.get_device_capability(i) == (7,5) for i in range(2))
model = get_peft_model(nn.Sequential(nn.Linear(4,4)), LoraConfig(target_modules=['0'], r=2))
assert any('lora_' in n and p.requires_grad for n,p in model.named_parameters())
print('torch',torch.__version__,'transformers',transformers.__version__,'peft',peft.__version__,'accelerate',accelerate.__version__)
for i in range(2): print(i, torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory/2**30,2),'GiB')
"""], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', 'pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'python-calamine', 'xlrd'], check=True)

## 4. Kiểm tra upload HF trước khi train
Bật Internet, thêm Kaggle Secret `HF_TOKEN` có quyền write và bật quyền truy cập secret cho notebook.
Kiểm tra repo private và upload một file probe nhỏ. Thực hiện trong tiến trình mới để cài đặt offline cũ của kernel không chặn HF. Nếu lỗi, dừng trước khi train và in nguyên nhân đã che token.
Đặt `HF_REPO_NAME=''` để tắt upload. Token chỉ nằm trong môi trường tiến trình.

In [ ]:
from datetime import datetime, timezone

def run_hf_upload(adapter_dir=None):
    token = os.environ.get('HF_TOKEN')
    if not token:
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            raise RuntimeError('Bật Internet và Kaggle Secret HF_TOKEN có quyền write cho notebook.') from None
    if not token or not token.strip():
        raise RuntimeError('HF_TOKEN rỗng; kiểm tra Kaggle Secrets.')
    os.environ['HF_TOKEN'] = token.strip()
    report = RUN_ROOT / ('hf_final_upload.json' if adapter_dir else 'hf_preflight.json')
    environment = dict(os.environ, HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0', HF_HUB_DISABLE_PROGRESS_BARS='1')
    command = [sys.executable, '-u', str(PROJECT / 'scripts/upload_hf_adapter.py'),
               '--repo', HF_REPO_NAME, '--prefix', os.environ['DATAMATE_HF_PREFIX'], '--report', str(report)]
    if adapter_dir:
        command += ['--adapter-dir', str(adapter_dir)]
    subprocess.run(command, env=environment, check=True)
    result = json.loads(report.read_text(encoding='utf-8'))
    print('HF:', result['status'], result['url'])
    return result

if HF_REPO_NAME:
    os.environ['HF_HUB_OFFLINE'] = '0'
    os.environ['TRANSFORMERS_OFFLINE'] = '0'
    os.environ['DATAMATE_HF_PREFIX'] = 'runs/' + RUN_NAME + '/' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    result = run_hf_upload()
    os.environ['DATAMATE_HF_REPO'] = result['repo_id']
else:
    os.environ.pop('DATAMATE_HF_REPO', None)
    os.environ.pop('DATAMATE_HF_PREFIX', None)
    print('HF upload tắt; lưu Kaggle Output để giữ adapter/checkpoint.')

## 5. Chọn weights Instruct cục bộ và kiểm tra đủ shard


In [ ]:
def is_qwen7b(path):
    try:
        cfg = json.loads((path / 'config.json').read_text(encoding='utf-8'))
        return (cfg.get('model_type') == 'qwen2' and cfg.get('hidden_size') == 3584
                and cfg.get('num_hidden_layers') == 28 and (path / 'tokenizer_config.json').is_file())
    except (OSError, ValueError):
        return False

if MODEL_DIR:
    model_path = Path(MODEL_DIR)
else:
    models = sorted({p.parent for p in Path('/kaggle/input').rglob('config.json') if is_qwen7b(p.parent)})
    if len(models) != 1:
        raise RuntimeError(f'Tìm được {len(models)} model Qwen7B: {models}. Điền MODEL_DIR ở cell 1.')
    model_path = models[0]
assert is_qwen7b(model_path), f'Không phải cấu hình Qwen7B hợp lệ: {model_path}'
index = model_path / 'model.safetensors.index.json'
if index.is_file():
    required = set(json.loads(index.read_text())['weight_map'].values())
else:
    required = {'model.safetensors'}
missing = [name for name in sorted(required) if not (model_path / name).is_file()]
assert not missing, f'Thiếu weights: {missing}'
MODEL_DIR = str(model_path.resolve())
print('Model:', MODEL_DIR)
print('Xác nhận Input này là Qwen2.5-Coder-7B-Instruct; cấu hình kiến trúc không chứng minh bản Instruct.')


## 6. Dựng prompt từ dataset gọn và kiểm tra token

In [ ]:
from collections import deque
import signal

def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=40)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1,
                              start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line); log.flush(); tail.append(line)
                status = process.wait()
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait()
                raise
    if status:
        raise RuntimeError(f'Lệnh lỗi (exit {status}). Log: {log_path}\n' + ''.join(tail))

# Rebuild once before FSDP; no gold or generated-code execution is needed here.
run_logged([sys.executable, '-u', 'scripts/build_sft_dataset.py',
            '--input', str(COMPACT_DATASET), '--data_dir', str(TRAIN_DATA), '--output', str(DATASET)],
           RUN_ROOT / 'prepare_context.log')
prepared = [json.loads(line) for line in DATASET.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(prepared) == 890
assert [row['id'] for row in prepared] == [row['id'] for row in records]
import re
assert all(not re.search(r'[A-Za-z]:[\\/]|/(?:content|kaggle|workspace)/', row['messages'][1]['content']) for row in prepared)
if shutil.disk_usage(WORK).free < 3 * 2**30:
    raise RuntimeError('Cần ít nhất 3 GiB đĩa trống trước train. Giữ adapter/checkpoint và dọn cache/snapshot cũ trước khi tiếp tục.')

BASE_ARGS = ['--model', MODEL_DIR, '--data', str(DATASET), '--max-seq-len', str(MAX_SEQ_LEN),
             '--epochs', str(EPOCHS), '--lr', str(LEARNING_RATE), '--grad-accum', str(GRAD_ACCUM)]
run_logged([sys.executable, '-u', 'scripts/train_sft_kaggle_2t4.py', *BASE_ARGS,
            '--output-dir', str(RUN_ROOT / 'preflight'), '--prepare-only'], RUN_ROOT / 'preflight.log')
report = json.loads((RUN_ROOT / 'preflight/dataset_report.json').read_text())
assert report['records'] == 890 and not report['overlength']
print('Đủ 890 mẫu. Tokens:', report['min_tokens'], '→', report['max_tokens'])

## 7. Smoke trên hai T4 — 2 optimizer steps với mẫu dài nhất

Cell này train thử, lưu adapter/optimizer rồi khôi phục checkpoint ngay trong phiên để kiểm tra đường save/load.
Nếu OOM hoặc lỗi thì dừng ở đây; chưa chạy full. Khi chạy lại smoke, xóa marker cũ bằng cách dùng RUN_NAME mới.


In [ ]:
LAUNCH = [sys.executable, '-m', 'accelerate.commands.launch', '--config_file',
          'configs/accelerate_kaggle_2t4_sft.yaml', 'scripts/train_sft_kaggle_2t4.py']
run_logged([*LAUNCH, *BASE_ARGS, '--output-dir', str(RUN_ROOT / 'smoke'), '--smoke'], RUN_ROOT / 'smoke.log')
smoke = json.loads((RUN_ROOT / 'smoke/completed.json').read_text())
assert smoke['status'] == 'passed' and smoke['checkpoint_restore_checked']
assert {r['rank'] for r in smoke['ranks']} == {0, 1}
assert all(r['steps'] == 2 and math.isfinite(r['loss']) and r['peak_vram_gib'] > 0 for r in smoke['ranks'])
print(json.dumps(smoke['ranks'], indent=2))

## 8. Train đủ 890 mẫu

Mặc định 3 epochs, LR 1e-4, LoRA r16/alpha32, effective batch 8. Đây là cấu hình khởi đầu, chưa phải hyperparameters được benchmark tối ưu.
Checkpoint mỗi 10 optimizer steps, giữ 2 bản và luôn lưu checkpoint cuối. Chỉ dùng checkpoint mình tạo.

Resume cùng phiên: đặt `RESUME_FROM_CHECKPOINT='latest'`. Phiên mới: gắn Output cũ thành Input và nhập đường dẫn `checkpoint-N` đầy đủ. Giữ nguyên model Input/path, data, settings và versions.


In [ ]:
assert (RUN_ROOT / 'smoke/completed.json').is_file(), 'Chạy smoke trước'
command = [*LAUNCH, *BASE_ARGS, '--output-dir', str(RUN_ROOT / 'train')]
if RESUME_FROM_CHECKPOINT:
    command += ['--resume', RESUME_FROM_CHECKPOINT]
run_logged(command, RUN_ROOT / 'train.log')
completed = json.loads((RUN_ROOT / 'train/completed.json').read_text())
assert completed['status'] == 'passed'
print('Adapter:', RUN_ROOT / 'train/final_adapter')
print(json.dumps(completed['ranks'], indent=2))


## 9. Upload final adapter và xác minh
Chạy lại cell này để thử upload lại; không train lại. Chỉ báo thành công khi hai file adapter đã có trên commit HF.


In [ ]:
# Run after full training, or set EVAL_ADAPTER_DIR to an existing adapter and skip train.
adapter_to_upload = Path(EVAL_ADAPTER_DIR) if EVAL_ADAPTER_DIR else RUN_ROOT / 'train/final_adapter'
assert (adapter_to_upload / 'adapter_model.safetensors').is_file(), f'Không tìm thấy adapter: {adapter_to_upload}'
if HF_REPO_NAME:
    uploaded = run_hf_upload(adapter_to_upload)
    assert uploaded['status'] == 'uploaded'
else:
    print('HF upload tắt. Lưu Kaggle Output trước khi đóng phiên.')

## 10. Cài runtime vLLM để đánh giá trên 2×T4
Upload adapter ở bước 9 trước khi cài/chạy đánh giá. Nếu thiếu đĩa cho môi trường vLLM, có thể dừng và đánh giá ở phiên khác với adapter đã lưu.
Runtime inference nằm trong `/tmp`; không đóng gói môi trường này vào Output.

In [ ]:
# Cài riêng để vLLM không thay torch/Transformers của phiên training.
EVAL_ENV = Path('/tmp/datamate_vllm_eval_env')
EVAL_PYTHON = EVAL_ENV / 'bin/python'
if not EVAL_ENV.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(EVAL_ENV)], check=True)

# Cài đặt nvidia-cusparselt-cu12 để cung cấp libcusparseLt.so.0
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'nvidia-cusparselt-cu12'], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(EVAL_PYTHON), 'install', '--no-cache-dir', 'nvidia-cusparselt-cu12'], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(EVAL_PYTHON), 'install', '--no-cache-dir', '-q',
                'vllm==0.10.2', 'transformers==4.57.1', 'peft==0.17.1', 'accelerate==1.11.0',
                'huggingface-hub==0.36.2', 'tokenizers==0.22.2',
                'pandas', 'pyyaml', 'python-calamine', 'openpyxl', 'xlrd', 'duckdb', 'tqdm'], check=True)

# Tìm và đăng ký libcusparseLt.so vào hệ thống qua ldconfig
find_res = subprocess.run("find / -name 'libcusparseLt.so*' 2>/dev/null", shell=True, capture_output=True, text=True)
so_files = [line.strip() for line in find_res.stdout.splitlines() if line.strip()]
for so_file in so_files:
    fname = Path(so_file).name
    for target_dir in ['/usr/lib', '/usr/local/lib', '/usr/lib/x86_64-linux-gnu']:
        if Path(target_dir).is_dir():
            target_path = Path(target_dir) / fname
            if not target_path.exists():
                try:
                    shutil.copy2(so_file, target_path)
                except Exception:
                    pass
subprocess.run(['ldconfig'], check=False)

subprocess.run([str(EVAL_PYTHON), '-c', "import torch, vllm; print('Inference OK:', 'PyTorch', torch.__version__, '| vLLM', vllm.__version__); assert torch.cuda.device_count() == 2"], check=True)


## 11. Chấm 30 câu pilot bằng vLLM
Pipeline tối đa 3 lượt/câu. PA, đúng/sai/abstain, thời gian và trace lưu tại `pilot_vllm/`.
Điểm pilot có giao với train; tốc độ/PA phải đo trên chính phiên Kaggle.

In [ ]:
ADAPTER_FOR_EVAL = Path(EVAL_ADAPTER_DIR) if EVAL_ADAPTER_DIR else RUN_ROOT / 'train/final_adapter'
assert (ADAPTER_FOR_EVAL / 'adapter_model.safetensors').is_file(), 'Cần adapter đã export đúng tên tensor.'
EVAL_ROOT = RUN_ROOT / 'pilot_vllm'
PILOT = bundle / 'evaluation/pilot_questions_30.jsonl'
GOLD = bundle / 'evaluation/pilot_gold_30.jsonl'
EVAL_DATA = bundle / 'evaluation/train'  # Đọc raw data trực tiếp từ Kaggle Input read-only.
training_ids = {row['id'] for row in records}
pilot_rows = [json.loads(line) for line in PILOT.read_text(encoding='utf-8').splitlines() if line.strip()]
overlap = len(training_ids & {row['id'] for row in pilot_rows})
print(f'Pilot: {len(pilot_rows)} câu; {overlap} ID có trong SFT. Điểm này có giao với train.')
# vLLM 0.10.2 còn V0 cho T4; engine/LoRA chạy hoàn toàn trong subprocess riêng.
eval_environment = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DATAMATE_ALLOW_ONLINE='0',
                        VLLM_USE_V1='0', VLLM_WORKER_MULTIPROC_METHOD='spawn', VLLM_NO_USAGE_STATS='1',
                        VLLM_ATTENTION_BACKEND='XFORMERS')
eval_environment.pop('PYTORCH_CUDA_ALLOC_CONF', None)
eval_environment.pop('PYTORCH_ALLOC_CONF', None)
command = [str(EVAL_PYTHON), '-u', 'scripts/evaluate_pilot_30.py',
           '--backend', 'vllm', '--tensor-parallel-size', '2', '--dtype', 'float16',
           '--gpu-memory-utilization', '0.90', '--enforce-eager', '--no-prefix-caching',
           '--max-model-len', '8192',
           '--adapter-path', str(ADAPTER_FOR_EVAL), '--base-model', MODEL_DIR,
           '--questions-file', str(PILOT), '--gold-file', str(GOLD),
           '--data-dir', str(EVAL_DATA), '--output-dir', str(EVAL_ROOT)]
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
with (RUN_ROOT / 'pilot_vllm.log').open('w', encoding='utf-8') as log:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          encoding='utf-8', errors='replace', env=eval_environment) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = process.wait()
if status:
    raise RuntimeError(f'vLLM evaluation lỗi (exit {status}); xem {RUN_ROOT / "pilot_vllm.log"}')
evaluation = json.loads((EVAL_ROOT / 'evaluation_report.json').read_text(encoding='utf-8'))
evaluation['training_id_overlap'] = overlap
evaluation['evaluation_scope'] = 'Train pilot with SFT ID overlap; not a fully held-out benchmark'
(EVAL_ROOT / 'evaluation_report.json').write_text(json.dumps(evaluation, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps({key: evaluation[key] for key in ['correct', 'wrong', 'abstain', 'penalized_accuracy',
                                                 'seconds_per_question', 'training_id_overlap']}, indent=2))


## 12. Đóng gói adapter và kết quả nhẹ
ZIP chứa final adapter, log/metrics và pilot; checkpoint resume vẫn ở Kaggle Output, không nhân bản vào ZIP. Nếu thiếu đĩa để tạo ZIP, giữ Output và dùng bản adapter đã upload HF.

In [ ]:
import zipfile

trained_adapter = Path(EVAL_ADAPTER_DIR) if EVAL_ADAPTER_DIR else RUN_ROOT / 'train/final_adapter'
assert (trained_adapter / 'adapter_model.safetensors').is_file()
assert (trained_adapter / 'adapter_config.json').is_file()
archive = Path('/kaggle/working') / (RUN_NAME + '_results.zip')
adapter_bytes = sum(p.stat().st_size for p in trained_adapter.rglob('*') if p.is_file())
if shutil.disk_usage('/kaggle/working').free < adapter_bytes + 512 * 2**20:
    print('Không đủ đĩa để tạo thêm ZIP. Adapter/checkpoint còn nguyên; giữ Kaggle Output hoặc dùng link HF.')
else:
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as output_zip:
        for source in sorted(trained_adapter.rglob('*')):
            if source.is_file():
                output_zip.write(source, Path('final_adapter') / source.relative_to(trained_adapter))
        for source in sorted(RUN_ROOT.rglob('*')):
            relative = source.relative_to(RUN_ROOT)
            if (source.is_file() and source != archive
                    and not any(part.startswith('checkpoint-') or part.endswith('-hub-staging') or part == 'final_adapter' for part in relative.parts)
                    and source.suffix in {'.json', '.jsonl', '.log', '.yaml'}):
                output_zip.write(source, relative)
    print('ZIP kết quả:', archive)
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))